# Alpaca paper bot: start here

**Step 1: Run the first cell.** It unpacks the included bot and runs 30 checks plus a fake-broker demo. No keys, purchases, account access or network calls are needed. Send the `OFFLINE TESTS` and `OFFLINE DEMO` lines back in our chat.

This notebook is a setup tool, not an unattended trading host. The bot package includes paper execution, persistent state, logs and Docker instructions. Neither code cell here enables orders or initializes a trading ledger.

**Step 2 is optional:** after your data entitlement changes, run the second cell to recheck the paper account and SIP. Use only hidden credential prompts. Your previous SIP 403 remains unresolved until that check succeeds.

Before paper execution, we still need a persistent worker/disk, matching SIP data, observation during market hours and a supervised paper session. Historical returns are not results achieved by this implementation. Entry limits, timing guards and partial fills can change results.

The included START_HERE.md contains deployment and troubleshooting instructions. The full source is embedded in the first cell as an archive; its SHA-256 checksum is verified before extraction.


In [ ]:
import base64, hashlib, io, pathlib, re, subprocess, sys, tempfile, zipfile
PAYLOAD = '...'
EXPECTED_SHA256 = '952898c9b18f6f5150de1914923e0def164548fdbb5a91629935b7efe74eaa92'
raw = base64.b64decode(PAYLOAD)
if hashlib.sha256(raw).hexdigest() != EXPECTED_SHA256:
    raise RuntimeError('Package checksum mismatch; stop here.')
BOT_DIR = pathlib.Path(tempfile.mkdtemp(prefix='alpaca-paper-bot-'))
with zipfile.ZipFile(io.BytesIO(raw)) as package:
    for item in package.infolist():
        name = pathlib.PurePosixPath(item.filename)
        if name.is_absolute() or '..' in name.parts:
            raise RuntimeError('Unsafe archive path; stop here.')
    package.extractall(BOT_DIR)
result = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests'], cwd=BOT_DIR, capture_output=True, text=True)
if result.returncode:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError('Offline checks failed. Share the error output; no credentials are involved.')
count = re.search(r'Ran (\d+) tests', result.stderr)
print('OFFLINE TESTS: PASS (' + (count.group(1) if count else 'all') + ' checks)')
demo = subprocess.run([sys.executable, '-m', 'paperbot', 'demo'], cwd=BOT_DIR, capture_output=True, text=True)
print(demo.stdout)
if demo.returncode:
    print(demo.stderr)
    raise RuntimeError('Demo failed.')
print('Next: send the PASS lines to our chat. No Alpaca orders have been placed.')


Optional read-only account and SIP check. This does not enable trading.

In [ ]:
# Optional read-only preflight. Run only after the first cell passes.
import getpass, os, subprocess, sys, warnings

def check_access():
    key = secret = None
    child_env = None
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('error', getpass.GetPassWarning)
            key = getpass.getpass('PAPER API key (hidden): ').strip()
            secret = getpass.getpass('PAPER secret (hidden): ').strip()
        if not key or not secret:
            print('Both paper credentials are required.'); return
        child_env = dict(os.environ, ALPACA_API_KEY=key, ALPACA_SECRET_KEY=secret)
        result = subprocess.run([sys.executable, '-m', 'paperbot', 'preflight'],
                                cwd=BOT_DIR, env=child_env, capture_output=True, text=True, timeout=120)
        print(result.stdout)
        if result.returncode and not result.stdout:
            print('Preflight could not complete. No orders were enabled.')
    except (getpass.GetPassWarning, EOFError, KeyboardInterrupt):
        print('Hidden input unavailable or cancelled. Do not paste keys into code.')
    except Exception:
        print('Preflight could not complete. Share only this status, never credentials.')
    finally:
        key = secret = None
        if child_env is not None:
            child_env.pop('ALPACA_API_KEY', None); child_env.pop('ALPACA_SECRET_KEY', None)

check_access()
